# Milestone 5: ML End-to-End with Insights

**Deadline:** `Mar 15, 2026`

1. Repeat the ML end-to-end process again with insights into the dataset.

## Import libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import OneHotEncoder

## Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

## Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

## Instructions for further questions

You are given two datasets:

* **Training dataset:** `df` (full training data)
* **Test dataset:** `test_df` (test data)

> **Note:** We are not going to use `test_df` anywhere in the below questions. However, in the project you will be required to process the test data as well. Due to this reason, we have given the task to perform preprocessing on the test data provided to you in the competition.

The **target variable** for the training dataset (`df`) is **`y`**.
Extract it from the training data and store it as `y`, then **drop the target column from the training data**.

Your task is to **preprocess the data so that it can be used for model training.**

---

### Step 1: Datetime Feature Engineering

For both **`df` and `test_df`**:

1. Convert the feature **`created_date`** to datetime format using:

   ```
   pd.to_datetime()
   ```

2. Extract the following features from `created_date`:

   * `year`
   * `month`
   * `hour`

3. Add these as **new features**.

4. After extracting these features, **drop the original `created_date` feature**.

---

### Step 2: Handle Missing Values

Perform the following preprocessing for **both datasets (`df` and `test_df`)**.

#### Categorical Features

Fill missing values with **"Unknown"** for the following features:

* `race`
* `religion`
* `gender`

#### Text Feature

Replace missing values in the **`comment`** feature with an **empty string `""`**.

#### Boolean Feature

Convert the **`disability`** feature to **integer type** using:

```
.astype(int)
```

---

### Step 3: Feature Transformation using `ColumnTransformer`

Use **`ColumnTransformer`** to process the features as follows.

#### Text Feature

Apply:

```
TfidfVectorizer(stop_words="english", max_features=5000)
```

to the feature:

```
comment
```

---

#### Categorical Features

Apply:

```
OneHotEncoder(drop="first", handle_unknown="ignore")
```

to the following features:

```
["race", "religion", "gender"]
```

---

#### Remaining Features

All remaining features should be passed through unchanged using:

```
remainder="passthrough"
```

This will keep numeric features such as:

* `year`
* `month`
* `hour`
* `disability`
* other numeric features

---

### Step 4: Transform the Data

Fit the `ColumnTransformer` **only on the training dataset (`df`)**.

Transform both datasets and store the results as:

```
X = preprocessor.fit_transform(df)
X_test = preprocessor.transform(test_df)
```

---

### Step 5: Create Train–Validation Split

Split the training features and labels `(X, y)` into **training and validation sets** using:

```
train_test_split
```

with the following parameters:

```
test_size = 0.2
random_state = 42
```

Store the outputs as:

```
X_train, X_val, y_train, y_val
```

---

### Final Output

After completing all steps, you should have the following variables ready for **model training**:

```
X_train
X_val
y_train
y_val
X_test
```

Use this **processed data** for all the questions below.

---

- [x] I have read and performed above preprocessing steps.

In [4]:
X = train.drop(columns=["label"])
y = train["label"]

In [5]:
X["created_date"] = pd.to_datetime(X["created_date"])

X["year"] = X["created_date"].dt.year
X["month"] = X["created_date"].dt.month
X["hour"] = X["created_date"].dt.hour

X.drop(columns=["created_date"], inplace=True, errors="ignore")

In [6]:
cols = ["race", "religion", "gender"]
X[cols] = X[cols].fillna("Unknown")

X["comment"] = X["comment"].fillna("")
X["disability"] = X["disability"].astype(int)

In [7]:
preprocessor = ColumnTransformer(
    [
        (
            "comment",
            TfidfVectorizer(stop_words="english", max_features=5000),
            "comment",
        ),
        ("cat_features", OneHotEncoder(drop="first", handle_unknown="ignore"), cols),
    ],
    remainder="passthrough",
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('comment', ...), ('cat_features', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [8]:
X = preprocessor.fit_transform(X)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

### Question 1

After applying TF-IDF vectorization on the comment column as per above instructions, what is the maximum number of text features that can be generated?

In [10]:
preprocessor.named_transformers_["comment"].max_features

5000

### Question 2

What are the number of features in `X_train`?

In [11]:
X_train.shape[1]

5031

### Question 3

What are the number of features in `X_val`?

In [12]:
X_val.shape[1]

5031

### Question 4

Train a **Multinomial Naive Bayes** model using the preprocessed data `X_train`, `y_train`. Generate predictions on the **validation dataset (X_val,y_val)** and evaluate the model using **accuracy score**. What **validation accuracy** (using X_val, y_val) does the Naive Bayes model achieve?

(Round the answer to 3 decimal places.)

In [13]:
naive = MultinomialNB()
naive.fit(X_train, y_train)

pred = naive.predict(X_val)
acc = accuracy_score(y_val, pred)
print(f"Validation Accuracy: {acc:.3f}")

Validation Accuracy: 0.808


### Question 5

What is the precision score for the label 3 from the classification report after performing the Multinomial Naive Bayes on validation set (X_val, y_val)?

(Round the answer to 2 decimal places.)

In [14]:
clf_report = classification_report(y_val, pred)
print(clf_report)

              precision    recall  f1-score   support

           0       0.90      0.89      0.89     22534
           1       0.45      0.81      0.58      3219
           2       0.83      0.71      0.77     12738
           3       0.52      0.27      0.35      1109

    accuracy                           0.81     39600
   macro avg       0.67      0.67      0.65     39600
weighted avg       0.83      0.81      0.81     39600



### Question 6

Train a **Logistic Regression** model using the training dataset with the following parameters:
```python
max_iter = 500, random_state = 42
```

Generate predictions on the **validation dataset** (X_val, y_val) and compute the **accuracy score**.
What **validation accuracy** does the Logistic Regression model achieve?

(Round the answer to 3 decimal places.)

In [15]:
log_reg = LogisticRegression(max_iter=500, random_state=42)
log_reg.fit(X_train, y_train)

pred_log = log_reg.predict(X_val)
acc_log = accuracy_score(y_val, pred_log)
print(f"Validation Accuracy: {acc_log:.3f}")

Validation Accuracy: 0.839


v:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Question 7

What **training accuracy** does the Logistic Regression model achieve? use X_train, y_train to calculate it.

(Round the answer to 3 decimal places.)

In [16]:
train_pred_log = log_reg.predict(X_train)
acc_log = accuracy_score(y_train, train_pred_log)
print(f"Validation Accuracy: {acc_log:.3f}")

Validation Accuracy: 0.841


### Question 8

What is the precision score for the label 1 from the classification report after performing the Logistic regression on  validation set (use X_val, y_val)?

(Round the answer to 2 decimal places.)

In [17]:
clf_report_log = classification_report(y_val, pred_log)
print(clf_report_log)

              precision    recall  f1-score   support

           0       0.96      0.95      0.95     22534
           1       0.55      0.00      0.00      3219
           2       0.69      0.93      0.79     12738
           3       0.00      0.00      0.00      1109

    accuracy                           0.84     39600
   macro avg       0.55      0.47      0.44     39600
weighted avg       0.81      0.84      0.80     39600



v:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
v:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
v:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

### Question 9

Perform **hyperparameter tuning** on the Logistic Regression model using **GridSearchCV** with **3-fold cross-validation**. Tune the parameter C using the values **0.1, 1, and 10**, and use the **liblinear solver**. Select the best model from GridSearch and evaluate it on the validation dataset.

Which value of **C out of 0.1, 1, 10** produces the best model?

In [18]:
grid_cv = GridSearchCV(
    estimator=LogisticRegression(solver="lbfgs"),
    param_grid={
        "C": [0.1, 1, 10],
    },
    cv=3,
    n_jobs=-1,
)
grid_cv.fit(X_train, y_train)

v:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are al

In [19]:
best_model = grid_cv.best_estimator_

In [20]:
grid_cv.best_params_["C"]

1

### Question 10

What **validation accuracy** do you obtain after hyperparameter tuning?

(Round the answer to 2 decimal places.)

In [21]:
best_pred = best_model.predict(X_val)
acc_best = accuracy_score(y_val, best_pred)
print(f"Validation Accuracy: {acc_best:.3f}")

Validation Accuracy: 0.839
